# CV Intelligent — Version Sans CrewAI

Pipeline IA direct : **3 appels API** au lieu de 18+.


In [5]:
!pip install openai pdfplumber python-dotenv pydantic jinja2 playwright -q
!playwright install chromium

D�finition de macro non valide.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\rayan\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


D�finition de macro non valide.


In [6]:
%pip install google-generativeai groq -q

D�finition de macro non valide.
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os, json, re, time
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
from typing import List, Optional
import pdfplumber
from openai import OpenAI, RateLimitError
import google.generativeai as genai
from groq import Groq as GroqClient

load_dotenv(find_dotenv(), override=True)

# ── Clients ───────────────────────────────────────────────────────────────────
openrouter_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
groq_client = GroqClient(api_key=os.getenv("GROQ_API_KEY"))

# ── Modèles par provider ──────────────────────────────────────────────────────
# Gemini direct (Google API) — quota gratuit : Flash-Lite 1000/j, Flash 250/j
GEMINI_MODELS = [
    "gemini-2.5-flash-lite",  # 1000 req/jour — priorité max
    "gemini-2.5-flash",       # 250 req/jour
]

# Groq direct — très rapide, quota gratuit généreux
GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "gemma2-9b-it",
]

# OpenRouter — uniquement les vrais modèles :free (0 crédit requis)
OPENROUTER_MODELS = [
    "openrouter/free",   # router auto — dernier recours
]


def appeler_llm(system: str, user: str, max_tokens: int = 2048) -> str:
    """
    Appel LLM avec fallback sur 3 providers gratuits dans l'ordre :
    1. Google Gemini direct (quota gratuit le plus généreux)
    2. Groq direct (rapide, gratuit)
    3. OpenRouter modèles :free (dernier recours)
    """
    # ── 1. Gemini (Google API directe) ───────────────────────────────────
    for model_name in GEMINI_MODELS:
        try:
            print(f"  -> Essai Gemini : {model_name}")
            model = genai.GenerativeModel(
                model_name=model_name,
                system_instruction=system
            )
            response = model.generate_content(
                user,
                generation_config=genai.GenerationConfig(
                    max_output_tokens=max_tokens,
                    temperature=0.1
                )
            )
            content = response.text
            if not content:
                print(f"  Reponse vide, essai suivant...")
                continue
            print(f"  OK avec Gemini {model_name}")
            return content
        except Exception as e:
            print(f"  Erreur Gemini {model_name}: {e}")
            time.sleep(1)

    # ── 2. Groq (API directe) ─────────────────────────────────────────────
    for model_name in GROQ_MODELS:
        try:
            print(f"  -> Essai Groq : {model_name}")
            response = groq_client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": user},
                ],
                temperature=0.1,
                max_tokens=max_tokens,
            )
            content = response.choices[0].message.content
            if not content:
                print(f"  Reponse vide, essai suivant...")
                continue
            print(f"  OK avec Groq {model_name}")
            return content
        except Exception as e:
            print(f"  Erreur Groq {model_name}: {e}")
            time.sleep(1)

    # ── 3. OpenRouter modèles :free ───────────────────────────────────────
    for model in OPENROUTER_MODELS:
        try:
            print(f"  -> Essai OpenRouter : {model}")
            response = openrouter_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": user},
                ],
                temperature=0.1,
                max_tokens=max_tokens,
            )
            content = response.choices[0].message.content
            if not content:
                print(f"  Reponse vide, essai suivant...")
                continue
            print(f"  OK avec OpenRouter {model}")
            return content
        except RateLimitError:
            print(f"  Rate-limited sur {model}, essai suivant...")
            time.sleep(2)
        except Exception as e:
            print(f"  Erreur OpenRouter {model}: {e}")
            time.sleep(2)

    raise RuntimeError("Tous les modeles sont indisponibles. Reessaie dans 1 minute.")


print("Configuration OK — Gemini + Groq + OpenRouter en fallback.")

C:\Users\rayan\AppData\Local\Temp\ipykernel_7916\3677473122.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Configuration OK — Gemini + Groq + OpenRouter en fallback.


In [8]:
from pydantic import BaseModel, field_validator
from pydantic.functional_validators import BeforeValidator
from typing import Annotated

def _coerce_list(v):
    """Convertit une string en liste si l'IA oublie les crochets."""
    if isinstance(v, str):
        return [s.strip() for s in v.split(',') if s.strip()]
    return v or []

# Type réutilisable : accepte liste OU string CSV
StrList = Annotated[List[str], BeforeValidator(_coerce_list)]


class Experience(BaseModel):
    titre:       Optional[str]  = None
    entreprise:  Optional[str]  = None
    duree:       Optional[str]  = None
    lieu:        Optional[str]  = None
    description: StrList        = []   # ← StrList au lieu de Optional[List[str]]

class Formation(BaseModel):
    diplome:       Optional[str] = None
    etablissement: Optional[str] = None
    annee:         Optional[str] = None
    description:   StrList       = []

class Projet(BaseModel):
    nom:          Optional[str] = None
    description:  StrList       = []
    technologies: StrList       = []  

class GroupeCompetence(BaseModel):
    categorie: str
    elements:  StrList = []

class ProfilCandidat(BaseModel):
    prenom:      Optional[str]                    = None
    nom:         Optional[str]                    = None
    email:       Optional[str]                    = None
    telephone:   Optional[str]                    = None
    ville:       Optional[str]                    = None
    linkedin:    Optional[str]                    = None
    github:      Optional[str]                    = None
    portfolio:   Optional[str]                    = None
    resume:      Optional[str]                    = None
    experiences: Optional[List[Experience]]       = []
    formations:  Optional[List[Formation]]        = []
    competences: Optional[List[GroupeCompetence]] = []
    projets:     Optional[List[Projet]]           = []
    langues:     StrList                          = []

class AnalyseOffre(BaseModel):
    titre_poste:            Optional[str] = None
    entreprise:             Optional[str] = None
    competences_requises:   StrList       = []
    competences_souhaitees: StrList       = []
    mots_cles_ats:          StrList       = []
    resume:                 Optional[str] = None

class ResultatFinal(BaseModel):
    profil_original:   Optional[ProfilCandidat] = None
    profil_optimise:   Optional[ProfilCandidat] = None
    analyse_offre:     Optional[AnalyseOffre]   = None
    lettre_motivation: Optional[str]            = None

print("Modeles OK.")

Modeles OK.


In [9]:
def lire_pdf(chemin: str) -> str:
    texte = ""
    with pdfplumber.open(chemin) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                texte += t + "\n"
    return texte.strip()


# ── CELLULE 4 : remplace extraire_json ──────────────────────────────────────
def extraire_json(texte: str) -> dict:
    texte = texte.strip()

    # Tentative 1 : bloc markdown ```json ... ```
    m = re.search(r"```(?:json)?\s*({[\s\S]*?})\s*```", texte)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass

    # Tentative 2 : premier objet JSON détecté dans le texte
    m = re.search(r"({[\s\S]*})", texte)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass

    # Tentative 3 : texte brut
    try:
        return json.loads(texte)
    except json.JSONDecodeError:
        # Affiche ce que le modèle a vraiment retourné pour déboguer
        print(f"\n[ERREUR] Reponse brute du modele (500 premiers chars) :\n{texte[:500]}\n")
        raise


print("Utilitaires OK.")

Utilitaires OK.


In [10]:
def generer_cv(chemin_cv_pdf: str, texte_offre: str) -> ResultatFinal:
    
    texte_offre = texte_offre[:6000]
    # ── Etape 1 : Lecture du PDF ──────────────────────────────────────────
    print("[1/4] Lecture du PDF...")
    texte_cv = lire_pdf(chemin_cv_pdf)
    if not texte_cv:
        raise ValueError("PDF vide ou illisible.")
    texte_cv_court = texte_cv[:12000]
    print(f"  PDF lu : {len(texte_cv)} caracteres extraits.")

    # ── Etape 2 : Extraction du profil (appel 1/4) ────────────────────────
    print("[2/4] Extraction du profil...")
    schema_profil  = json.dumps(ProfilCandidat.model_json_schema(), ensure_ascii=False)
    reponse_profil = appeler_llm(
        system=(
            "Tu es un parseur de CV expert. "
            "Retourne UNIQUEMENT un objet JSON valide, sans texte autour.\n"
            "CRITIQUE : Tu dois extraire l'EXHAUSTIVITE absolue du CV. "
            "N'oublie AUCUN projet, AUCUNE experience, et AUCUNE competence. "
            "Garde les descriptions longues et detaillees originales, ne les resume pas."
        ),
        user="Schema attendu :\n" + schema_profil + "\n\nCV a analyser :\n" + texte_cv_court,
        max_tokens=4000
    )
    print(f"  [DEBUG extraction] {reponse_profil[:300]}")  # voir ce que le modele a retourné
    profil_original = ProfilCandidat(**extraire_json(reponse_profil))

    # Validation : profil vide = le modele n'a pas suivi les instructions JSON
    if not profil_original.nom and not profil_original.experiences and not profil_original.projets:
        raise RuntimeError(
            "L'extraction du CV a retourne un profil vide.\n"
            "Cause probable : openrouter/free a choisi un modele trop faible.\n"
            "Solution : relance la cellule (un autre modele sera selectionne)."
        )
    print(f"  Profil extrait : {profil_original.prenom} {profil_original.nom} "
          f"— {len(profil_original.experiences or [])} exp, "
          f"{len(profil_original.projets or [])} projets")

    # ── Etape 3 : Analyse de l'offre (appel 2/4) ─────────────────────────
    print("[3/4] Analyse de l'offre...")
    schema_offre  = json.dumps(AnalyseOffre.model_json_schema(), ensure_ascii=False)
    reponse_offre = appeler_llm(
        system="Tu es un analyste RH. Retourne UNIQUEMENT un objet JSON valide, sans texte autour.",
        user="Schema attendu :\n" + schema_offre + "\n\nOffre d'emploi :\n" + texte_offre,
        max_tokens=2048
    )
    analyse_offre = AnalyseOffre(**extraire_json(reponse_offre))

    # Variables partagées entre les étapes 4a et 4b
    profil_json = profil_original.model_dump_json(indent=2)
    offre_json  = analyse_offre.model_dump_json(indent=2)

    # ── Etape 4a : Adaptation du CV (appel 3/4) ───────────────────────────
    print("[4/4] Adaptation du CV...")
    reponse_cv = appeler_llm(
        system=(
            "Tu es un expert RH et redacteur de CV. "
            "Retourne UNIQUEMENT un objet JSON valide representant le ProfilCandidat optimise "
            "pour l'offre d'emploi. Aucun texte autour du JSON.\n"
            "CRITIQUE :\n"
            "- Conserve TOUS les projets et TOUTES les experiences du profil original.\n"
            "- Ne supprime rien. Garde les descriptions detaillees."
        ),
        user=(
            "Profil candidat original :\n" + profil_json +
            "\n\nAnalyse de l'offre :\n" + offre_json +
            "\n\nSchema ProfilCandidat attendu :\n" + schema_profil
        ),
        max_tokens=4000
    )
    profil_optimise = ProfilCandidat(**extraire_json(reponse_cv))

    # ── Etape 4b : Lettre de motivation (appel 4/4) ───────────────────────
    print("[4/4] Generation de la lettre de motivation...")
    reponse_lettre = appeler_llm(
        system=(
            "Tu es un expert en redaction de lettres de motivation. "
            "Redige une lettre de motivation professionnelle de 200 a 300 mots "
            "adaptee au poste et au profil du candidat. "
            "Retourne UNIQUEMENT le texte de la lettre, sans JSON, sans titre, sans mise en forme."
        ),
        user=(
            "Profil du candidat :\n" + profil_optimise.model_dump_json(indent=2) +
            "\n\nPoste vise :\n" + offre_json
        ),
        max_tokens=1024
    )
    lettre = reponse_lettre.strip()

    print("Pipeline termine avec succes !")
    return ResultatFinal(
        profil_original=profil_original,
        profil_optimise=profil_optimise,
        analyse_offre=analyse_offre,
        lettre_motivation=lettre
    )


print("Pipeline pret.")

Pipeline pret.


In [1]:
CHEMIN_CV = r"C:/projects/PFE FR 1st YEAR/PFE CODE/CV_Rayane_Berrada.pdf"

OFFRE = (
    """About the job
Description du poste

A la Direction de l’Innovation, nous menons des projets de recherche sur des champs d’expérimentation très larges et multi-sectoriels.

Ces projets innovants, développés en équipes, sont encadrés par nos experts au sein des ALTEN Labs (IDF, Toulouse, Grenoble, Rennes et Sophia Antipolis), et tentent de répondre aux enjeux de nos clients en leur fournissant des solutions technologiques originales et disruptives.


Au sein de notre Lab de Sèvres, vous serez accompagné(e) par un Pilote Innovation (Chef de projet) pour vous permettre de développer vos compétences sur les activités du projet suivant.


Projet : Outil d'automatisation pour le spatial


Développement Front / Back de démonstrateurs avec Python, Java, React
Comparaison et ré-écriture automatique de processus
Gestion de Base de données diverses 

Qualifications

Vous êtes étudiant(e) en dernière année d’École d’Ingénieur à la recherche d’un stage de fin d’études et vous avez suivi une spécialité en Génie Logiciel et/ou systèmes.


Vous justifiez de connaissances en développement fullstack (Java/JEE ou Javascript/Angular, Spring, ReactJS, Python, React, SQL/NoSQL) et maîtrisez plusieurs outils associés tels que Figma, Intellij, Maven Git, Docker que vous avez su mettre en application lors de votre formation.



Des connaissances en modélisation et simulation ainsi que des notions en ontologie et/ou Intelligence Artificielle seraient un plus.


Réactif(ve), rigoureux(se), autonome et doté(e) du sens du service, vous souhaitez évoluer dans un environnement challengeant.




Informations supplémentaires

Comment rejoindre un de nos Labs ? 


Partagez-nous votre CV pour que l’on découvre votre parcours et ce qui vous anime.
Échangez avec un chargé de recrutement lors d’une première discussion afin de mieux comprendre vos attentes.
Rencontrez un pilote innovation pour parler concret, projets, idées… et voir si la magie opère.
Rejoindre nos ALTEN Labs, c’est s’immerger au cœur de notre culture de l’innovation. Vous aurez l’opportunité de développer vos compétences sur des sujets concrets, en collaborant au sein d’une équipe projet dynamique.


Nos équipes d’experts vous accompagneront pour devenir acteur de votre projet, dans un environnement multiculturel et pluridisciplinaire. Vous bénéficierez ainsi d’une expérience enrichissante, ouverte à tous les secteurs de l’ingénierie, en France comme à l’international.


Détails du stage


Durée de stage : 6 mois (Une embauche pourrait être envisagée à l’issue du stage)
Lieu de stage : Lab de Sèvres
Gratification de stage : 1300 euros brut/mois + Tickets restaurants + Prise en charge titre de transport
Vous vous reconnaissez dans ce descriptif ? Alors n’attendez plus ! Rejoignez une équipe qui place l’innovation au cœur de ses actions et où chaque talent peut laisser son empreinte."""
)

resultat = generer_cv(CHEMIN_CV, OFFRE)

print("\n" + "=" * 60)
print("Profil original extrait du CV :")
print(json.dumps(resultat.profil_original.model_dump(), ensure_ascii=False, indent=2))
print("\nProfil optimise pour l'offre :")
print(json.dumps(resultat.profil_optimise.model_dump(), ensure_ascii=False, indent=2))
print("\nLettre de motivation :")
print(resultat.lettre_motivation)

NameError: name 'generer_cv' is not defined

In [12]:
import os, json, subprocess
from jinja2 import Template
from pathlib import Path
import pdfplumber


# CSS injecte dynamiquement selon le niveau de tightening.
# Cle 0 = pas de surcharge (template deja optimise).
# Cle 1 = plateau modere : marges reduites.
# Cle 2 = plateau severe : marges + fontes reduites.
TIGHTENING_CSS = {0: "", 1: '<style>\n  .col-left  { padding: 15px 12px !important; gap: 8px !important; }\n  .col-right { padding: 15px 15px !important; }\n  .right-item-bloc { margin-bottom: 8px !important; line-height: 1.35 !important; }\n  .section-right-title { margin-bottom: 6px !important; padding: 4px !important; }\n  .section-left-title  { margin-bottom: 6px !important; padding: 4px !important; }\n  .header h2 { margin-bottom: 6px !important; }\n  ul li { margin-bottom: 1px !important; }\n</style>', 2: '<style>\n  .col-left  { padding: 12px 10px !important; gap: 6px !important; }\n  .col-right { padding: 12px 12px !important; }\n  .right-item-bloc { margin-bottom: 6px !important; font-size: 11px !important; line-height: 1.3 !important; }\n  .right-item-bloc strong { font-size: 12px !important; margin-bottom: 0 !important; }\n  .right-item-bloc .subtitle2 { margin-bottom: 3px !important; }\n  .section-right-title { margin-bottom: 5px !important; padding: 3px !important; font-size: 11px !important; }\n  .section-left-title  { margin-bottom: 5px !important; padding: 3px !important; font-size: 11px !important; }\n  .section-content { font-size: 10px !important; line-height: 1.3 !important; }\n  ul li { margin-bottom: 0 !important; }\n  .technos { margin-top: 1px !important; }\n</style>'}


def renderiser_cv_html(profil_d: dict, tightening: int = 0) -> str:
    """
    tightening=0 : espacement par defaut.
    tightening=1 : marges reduites (plateau detecte).
    tightening=2 : marges + fontes reduites (plateau severe).
    """
    job_title = (
        profil_d.get("experiences")
        and profil_d["experiences"]
        and profil_d["experiences"][0].get("titre")
    ) or None
    template_html = Path("template_cv.html").read_text(encoding="utf-8")
    html = Template(template_html).render(
        nom=profil_d.get("nom", ""),
        prenom=profil_d.get("prenom", ""),
        email=profil_d.get("email", ""),
        telephone=profil_d.get("telephone", ""),
        ville=profil_d.get("ville", ""),
        linkedin=profil_d.get("linkedin", ""),
        github=profil_d.get("github", ""),
        portfolio=profil_d.get("portfolio", ""),
        resume=profil_d.get("resume", ""),
        experiences=profil_d.get("experiences", []),
        formations=profil_d.get("formations", []),
        competences=profil_d.get("competences", []),
        projets=profil_d.get("projets", []),
        langues=profil_d.get("langues", []),
        job_title=job_title,
    )
    # Injecte le CSS de tightening dans le placeholder du template
    html = html.replace("<!-- TIGHTENING_PLACEHOLDER -->", TIGHTENING_CSS.get(tightening, ""))
    return html


def construire_html_lettre(profil_d: dict, lettre: str) -> str:
    nom_complet = f"{profil_d.get('prenom', '')} {profil_d.get('nom', '')}".strip()
    paragraphes = "".join(
        f"<p>{p.strip()}</p>" for p in lettre.split("\n") if p.strip()
    )
    return f"""<!DOCTYPE html>
<html lang="fr"><head><meta charset="UTF-8"><style>
  body      {{ font-family: "Segoe UI", sans-serif; margin: 60px; color: #1a1a1a; }}
  .nom      {{ font-size: 22px; font-weight: bold; color: #0f344a; }}
  .contact  {{ font-size: 12px; color: #555; margin-top: 4px; }}
  h1        {{ font-size: 16px; color: #0f344a; margin: 30px 0 20px;
               text-transform: uppercase; border-bottom: 2px solid #0f344a; padding-bottom: 8px; }}
  p         {{ font-size: 13px; line-height: 1.8; margin-bottom: 14px; text-align: justify; }}
  .signature{{ margin-top: 40px; font-weight: bold; }}
</style></head><body>
  <div class="nom">{nom_complet}</div>
  <div class="contact">{profil_d.get('email','')} | {profil_d.get('telephone','')} | {profil_d.get('ville','')}</div>
  <h1>Lettre de Motivation</h1>
  {paragraphes}
  <div class="signature">{nom_complet}</div>
</body></html>"""


def generer_pdfs(html_cv: str, html_lettre: str) -> None:
    Path("output/cv_optimise.html").write_text(html_cv, encoding="utf-8")
    Path("output/lettre_motivation.html").write_text(html_lettre, encoding="utf-8")
    script = """
from playwright.sync_api import sync_playwright
from pathlib import Path

def render(html_path, pdf_path):
    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page()
        page.goto(Path(html_path).resolve().as_uri())
        page.pdf(path=pdf_path, format="A4", print_background=True,
                 margin={"top":"0","bottom":"0","left":"0","right":"0"})
        browser.close()

render("output/cv_optimise.html",       "output/cv_optimise.pdf")
render("output/lettre_motivation.html", "output/lettre_motivation.pdf")
"""
    Path("generer_pdf.py").write_text(script, encoding="utf-8")
    res = subprocess.run(["python", "generer_pdf.py"], capture_output=True, text=True)
    Path("generer_pdf.py").unlink(missing_ok=True)
    if res.returncode != 0:
        print("Erreur generation PDF:", res.stderr)


# ── Sauvegarde initiale ───────────────────────────────────────────────────────
os.makedirs("output", exist_ok=True)
with open("output/profil_original.json", "w", encoding="utf-8") as f:
    json.dump(resultat.profil_original.model_dump(), f, ensure_ascii=False, indent=2)
with open("output/lettre_motivation.txt", "w", encoding="utf-8") as f:
    f.write(resultat.lettre_motivation or "")

# ── Boucle de verification (Objectif : 1 page MAX) ────────────────────────────
profil_courant  = resultat.profil_optimise
schema_profil   = json.dumps(ProfilCandidat.model_json_schema(), ensure_ascii=False)
max_tentatives  = 15
lettre_html     = construire_html_lettre(
    resultat.profil_optimise.model_dump(), resultat.lettre_motivation or ""
)

chars_precedent = None   # détection du plateau
nb_plateau      = 0      # combien de fois consécutives chars_p2 n'a pas changé

print("Lancement de la boucle de verification (Objectif : 1 page MAX)...")

for tentative in range(1, max_tentatives + 1):
    print(f"\nTentative {tentative}/{max_tentatives} — Generation du PDF...")

    generer_pdfs(renderiser_cv_html(profil_courant.model_dump()), lettre_html)

    try:
        with pdfplumber.open("output/cv_optimise.pdf") as pdf:
            nb_pages  = len(pdf.pages)
            if nb_pages > 1:
                texte_p2  = pdf.pages[1].extract_text() or ""
                chars_p2  = len(texte_p2.strip())
                lignes_p2 = len([l for l in texte_p2.split("\n") if l.strip()])
            else:
                texte_p2 = ""
                chars_p2 = lignes_p2 = 0
    except Exception as e:
        print("Erreur lecture PDF:", e)
        break

    print(f"-> CV genere : {nb_pages} page(s)." +
          (f" (page 2 : {chars_p2} chars, {lignes_p2} lignes)" if nb_pages > 1 else ""))

    if nb_pages == 1:
        print("CV tient sur 1 page.")
        break

    if tentative == max_tentatives:
        print("Nombre max de tentatives atteint. On garde ce resultat.")
        break

    # ── Détection du plateau ──────────────────────────────────────────────
    if chars_p2 == chars_precedent:
        nb_plateau += 1
    else:
        nb_plateau = 0
    chars_precedent = chars_p2

    # ── Instruction selon le niveau de blocage ────────────────────────────
    if nb_plateau >= 2:
        # Plateau détecté : on cible explicitement le contenu de la page 2
        instruction_reduction = (
            f"ATTENTION : Tu as déjà produit ce même résultat {nb_plateau} fois. "
            "Change de stratégie : identifie les sections ci-dessous qui correspondent "
            "au contenu de la page 2 et raccourcis-les de facon plus agressive (50-70%). "
            "Tu peux fusionner des points ou raccourcir les descriptions plus fortement."
        )
    else:
        instruction_reduction = (
            "Raccourcis les descriptions des sections les plus longues "
            "en gardant toutes les informations essentielles pour le poste."
        )

    print(f"CV trop long. Appel IA (plateau={nb_plateau})...")

    reponse = appeler_llm(
        system=(
            f"Tu es un expert en redaction de CV. Le CV fait {nb_pages} pages.\n"
            f"Voici EXACTEMENT le texte qui deborde sur la page 2 :\n"
            f"---\n{texte_p2}\n---\n"
            f"({chars_p2} caracteres, {lignes_p2} lignes)\n\n"
            "L'objectif ABSOLU est de tenir sur 1 seule page A4.\n"
            "Identifie dans le profil les sections qui correspondent au texte de la page 2 "
            "et reduis SPECIFIQUEMENT ces sections.\n"
            f"{instruction_reduction}\n\n"
            "REGLES :\n"
            "- Conserve TOUTES les experiences et TOUS les projets.\n"
            "- Garde les informations cles (technologies, realisations, chiffres).\n"
            "- Formulations courtes et directes.\n"
            "Retourne UNIQUEMENT le profil mis a jour au format JSON valide."
        ),
        user=(
            "Profil complet a raccourcir :\n" + profil_courant.model_dump_json() +
            "\n\nSchema attendu :\n" + schema_profil
        ),
        max_tokens=4000
    )
    profil_courant = ProfilCandidat(**extraire_json(reponse))

print("\nTermine ! Fichiers dans le dossier output/")

Lancement de la boucle de verification (Objectif : 1 page MAX)...

Tentative 1/15 — Generation du PDF...
-> CV genere : 2 page(s). (page 2 : 395 chars, 9 lignes)
CV trop long. Appel IA (plateau=0)...
  -> Essai Gemini : gemini-2.5-flash-lite
  OK avec Gemini gemini-2.5-flash-lite

Tentative 2/15 — Generation du PDF...
-> CV genere : 2 page(s). (page 2 : 31 chars, 1 lignes)
CV trop long. Appel IA (plateau=0)...
  -> Essai Gemini : gemini-2.5-flash-lite
  OK avec Gemini gemini-2.5-flash-lite

Tentative 3/15 — Generation du PDF...
-> CV genere : 2 page(s). (page 2 : 31 chars, 1 lignes)
CV trop long. Appel IA (plateau=1)...
  -> Essai Gemini : gemini-2.5-flash-lite
  OK avec Gemini gemini-2.5-flash-lite

Tentative 4/15 — Generation du PDF...
-> CV genere : 2 page(s). (page 2 : 31 chars, 1 lignes)
CV trop long. Appel IA (plateau=2)...
  -> Essai Gemini : gemini-2.5-flash-lite
  OK avec Gemini gemini-2.5-flash-lite

Tentative 5/15 — Generation du PDF...
-> CV genere : 2 page(s). (page 2 : 31